In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Handling Missing Values in Pandas - Mini Project Solution\n",
    "\n",
    "This notebook provides a complete walkthrough of the final mini-project from the tutorial. It demonstrates a practical workflow for handling missing values in the `sensor_log.csv` dataset, including decisions, code, and reasoning. All steps are self-contained and executable in Colab or Jupyter.\n",
    "\n",
    "### Learning Recap\n",
    "This builds on the tutorial's strategies: detection, dropping, simple imputation (mean/median/constant), and time-series methods (ffill, bfill, interpolation). Here, we apply them end-to-end for sensor data analysis."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Load sensor_log.csv into a New DataFrame\n",
    "\n",
    "We'll recreate the dataset for portability (exact match to tutorial). Shape: 10 rows, 4 columns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "# Recreate for portability (matches exact dataset structure)\n",
    "data = {\n",
    "    'timestamp': [\n",
    "        '2025-10-01 08:00:00', '2025-10-01 08:00:10', '2025-10-01 08:00:20', '2025-10-01 08:00:30', '2025-10-01 08:01:00',\n",
    "        '2025-10-01 08:02:00', '2025-10-01 08:03:00', '2025-10-01 08:04:00', '2025-10-01 08:05:00', '2025-10-01 08:05:30'\n",
    "    ],\n",
    "    'temperature_c': [24.5, 24.7, 24.6, np.nan, 24.9, 25.0, 25.3, 25.1, np.nan, 25.5],\n",
    "    'humidity_pct': [55.2, 55.0, 55.1, 54.9, 54.8, 54.7, 54.7, np.nan, 54.9, 54.9],\n",
    "    'voltage_v': [3.70, 3.69, np.nan, 3.68, 3.68, 3.67, 3.67, 3.66, 3.66, 3.65]\n",
    "}\n",
    "df = pd.DataFrame(data)\n",
    "\n",
    "# First look\n",
    "print('Shape:', df.shape)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Summarise Missing Values per Column (Counts and Percentages)\n",
    "\n",
    "Total missing: 4 out of 30 numeric values (~13%)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "missing_counts = df.isna().sum()\n",
    "missing_pct = (df.isna().mean() * 100).round(1)\n",
    "missing_summary = pd.DataFrame({'Counts': missing_counts, 'Percent': missing_pct})\n",
    "missing_summary"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: Decide Which Columns or Rows (If Any) to Drop\n",
    "\n",
    "**Decision:** No rows or columns will be dropped.\n",
    "\n",
    "**Justification:** The missing values are minimal (max 20% in `temperature_c`, averaging 10% across numerics) and appear randomly scattered across timestamps, likely due to brief sensor failures rather than systematic issues (e.g., no clustering at specific times like low battery). Dropping rows with any NaN would eliminate 4 rows (40% of the dataset), creating gaps in the time series that could distort trend visualizations or downstream analysis (e.g., hourly averages). Similarly, dropping columns isn't warranted—no column is >20% missing, and all (temp, humidity, voltage) are core to sensor monitoring. Retention maximizes data utility while imputation can handle the gaps safely."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: Choose and Apply an Imputation Strategy\n",
    "\n",
    "**Chosen Strategy:** Forward fill (`ffill()`) for this time series data.\n",
    "\n",
    "**Why?** Sensor readings evolve gradually (e.g., temp rises ~0.1°C every 10–20s), so propagating the last valid value forward mimics real-world continuity during glitches without introducing bias from global averages or future peeks (like backward fill). It's simple, preserves order, and avoids assuming linearity (unlike interpolation). Applied after converting `timestamp` to datetime and setting it as index."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_ts = df.copy()\n",
    "df_ts['timestamp'] = pd.to_datetime(df_ts['timestamp'])\n",
    "df_ts = df_ts.set_index('timestamp')\n",
    "df_imputed = df_ts.ffill()\n",
    "\n",
    "# Check: No NaNs left\n",
    "print('Missing after imputation:', df_imputed.isna().sum().sum())\n",
    "df_imputed.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5: Compare Key Summary Statistics (Mean, Min, Max) Before and After Imputation\n",
    "\n",
    "Focus on numeric columns. Original stats ignore NaNs by default.\n",
    "\n",
    "**Insights:** Means shift minimally (e.g., temp drops 0.02°C due to carrying early cooler values forward), while min/max remain unchanged—imputation adds no extremes. This confirms low distortion, ideal for stable sensor baselines."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "numeric_cols = ['temperature_c', 'humidity_pct', 'voltage_v']\n",
    "orig_stats = df[numeric_cols].agg(['mean', 'min', 'max'])\n",
    "imputed_stats = df_imputed[numeric_cols].agg(['mean', 'min', 'max'])\n",
    "comparison = pd.concat([orig_stats, imputed_stats], keys=['Original', 'Imputed'], axis=1).round(3)\n",
    "comparison"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6: Short Paragraph Explaining Decisions\n",
    "\n",
    "For this sensor dataset, I opted not to drop any rows or columns due to the low and random incidence of missing values (≤20% per column), which likely stem from transient hardware issues rather than structural flaws; preserving the full 10-row timeline avoids biasing time-based insights like gradual warming trends. Forward fill was selected for imputation as it aligns with the sequential, slowly varying nature of environmental metrics—carrying prior readings forward simulates continuity during glitches without over-relying on dataset-wide statistics (like mean fill) or hindsight (backward fill), ensuring imputed values stay contextually grounded. Post-imputation stats show negligible shifts in means (e.g., <0.1% change) and identical ranges, validating the approach's minimal impact while enabling robust downstream tasks like plotting or alerting on voltage dips. This strategy balances data retention with realism, a practical choice for real-time IoT monitoring where completeness trumps perfection."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Next Steps\n",
    "Try this on your own dataset! Experiment with other strategies (e.g., median fill) and visualize changes with `df.plot()`. This reasoning—explore, decide, apply, validate—is key for data analysts."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}